# Day 3: Extract Operational Manufacturing Data

This notebook connects to the NorDex manufacturing database, examines the available table, extracts a defined production period, exports the result to CSV and validates the exported dataset.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

In [2]:
database_path = Path("ShiftData.db")

print("Database exists:", database_path.exists())
print(
    "Database size:",
    round(database_path.stat().st_size / (1024 ** 2), 2),
    "MB"
)

Database exists: True
Database size: 69.82 MB


In [3]:
connection = sqlite3.connect(database_path)

print("Database connection successful.")

Database connection successful.


In [4]:
tables_query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

available_tables = pd.read_sql_query(
    tables_query,
    connection
)

available_tables

,name
0,ShiftPerformance


In [5]:
schema_query = """
PRAGMA table_info(ShiftPerformance);
"""

table_schema = pd.read_sql_query(
    schema_query,
    connection
)

table_schema

,cid,name,type,notnull,dflt_value,pk
0,0,shift_id,INTEGER,0,None,0
1,1,shift_name,TEXT,0,None,0
2,2,start_time,TEXT,0,None,0
3,3,end_time,TEXT,0,None,0
4,4,supervisor_id,TEXT,0,None,0
5,5,production_id,INTEGER,0,None,0
6,6,date,TEXT,0,None,0
7,7,units_produced,INTEGER,0,None,0
8,8,defect_count,INTEGER,0,None,0
9,9,cycle_time_avg,REAL,0,None,0


In [6]:
database_summary_query = """
SELECT
    COUNT(*) AS total_records,
    MIN(date) AS earliest_date,
    MAX(date) AS latest_date,
    COUNT(DISTINCT machine_id) AS number_of_machines,
    COUNT(DISTINCT operator_id) AS number_of_operators
FROM ShiftPerformance;
"""

database_summary = pd.read_sql_query(
    database_summary_query,
    connection
)

database_summary

,total_records,earliest_date,latest_date,number_of_machines,number_of_operators
0,296334,2024-01-01,2024-03-31,50,110


## Extraction Criteria

The complete database contains 296,334 records and is too large for the submission platform's 5 MB upload limit. A three-day operational extract covering 1–3 January 2024 was therefore selected.

All 31 source columns and all shifts and machine statuses within this period were retained. This preserves the raw database structure while producing a manageable file for submission and downstream analysis.

In [7]:
start_date = "2024-01-01"
end_date = "2024-01-03"

extraction_query = """
SELECT *
FROM ShiftPerformance
WHERE date BETWEEN ? AND ?
ORDER BY
    date,
    shift_id,
    production_id;
"""

raw_manufacturing_data = pd.read_sql_query(
    extraction_query,
    connection,
    params=(start_date, end_date)
)

raw_manufacturing_data.head()

,shift_id,shift_name,start_time,end_time,supervisor_id,production_id,date,units_produced,defect_count,cycle_time_avg,...,issue_type,maintenance_downtime,resolved_by,qc_id,defect_type,severity,inspection_result,temperature,humidity,timestamp
0,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,...,NaN,NaN,NaN,1,Material,Minor,Rework,22.3,49.9,2024-01-01 08:00:00
1,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,...,NaN,NaN,NaN,2,Assembly,Minor,Accepted,22.3,49.9,2024-01-01 08:00:00
2,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,...,NaN,NaN,NaN,3,Dimensional,Minor,Rework,22.3,49.9,2024-01-01 08:00:00
3,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,...,NaN,NaN,NaN,4,Dimensional,Minor,Rework,22.3,49.9,2024-01-01 08:00:00
4,1,Morning,06:00:00.0000000,14:00:00.0000000,SUP_01,1,2024-01-01,929,20,35.65,...,NaN,NaN,NaN,5,Cosmetic,Minor,Rework,22.3,49.9,2024-01-01 08:00:00


In [8]:
extraction_validation = pd.Series({
    "Extracted rows":
        raw_manufacturing_data.shape[0],

    "Extracted columns":
        raw_manufacturing_data.shape[1],

    "Exact duplicate rows":
        raw_manufacturing_data.duplicated().sum(),

    "Earliest extracted date":
        raw_manufacturing_data["date"].min(),

    "Latest extracted date":
        raw_manufacturing_data["date"].max(),

    "Machines represented":
        raw_manufacturing_data["machine_id"].nunique(),

    "Operators represented":
        raw_manufacturing_data["operator_id"].nunique()
})

extraction_validation

Extracted rows                  10117
Extracted columns                  31
Exact duplicate rows                0
Earliest extracted date    2024-01-01
Latest extracted date      2024-01-03
Machines represented               50
Operators represented             104
dtype: object

In [9]:
print("Shift coverage:")

display(
    raw_manufacturing_data[
        "shift_name"
    ].value_counts()
)

print("Machine-status coverage:")

display(
    raw_manufacturing_data[
        "machine_status"
    ].value_counts()
)

Shift coverage:


shift_name
Night      3552
Evening    3503
Morning    3062
Name: count, dtype: int64

Machine-status coverage:


machine_status
Operational    6614
Issues         3503
Name: count, dtype: int64

In [10]:
raw_missing_report = pd.DataFrame({
    "missing_count":
        raw_manufacturing_data.isna().sum(),

    "missing_percentage":
        (
            raw_manufacturing_data.isna().mean()
            * 100
        ).round(2)
})

raw_missing_report = raw_missing_report.query(
    "missing_count > 0"
)

raw_missing_report

,missing_count,missing_percentage
maintenance_id,6223,61.51
issue_type,6223,61.51
maintenance_downtime,6223,61.51
resolved_by,6223,61.51
temperature,1205,11.91
humidity,1205,11.91
timestamp,1205,11.91


In [11]:
output_file = Path(
    "nordex_operational_extract_2024-01-01_to_2024-01-03.csv"
)

raw_manufacturing_data.to_csv(
    output_file,
    index=False
)

print("Export completed:", output_file.name)

Export completed: nordex_operational_extract_2024-01-01_to_2024-01-03.csv


In [12]:
exported_data_check = pd.read_csv(
    output_file
)

file_size_mb = (
    output_file.stat().st_size
    / (1024 ** 2)
)

export_validation = pd.Series({
    "File exists":
        output_file.exists(),

    "File size in MB":
        round(file_size_mb, 2),

    "Rows in exported CSV":
        exported_data_check.shape[0],

    "Columns in exported CSV":
        exported_data_check.shape[1],

    "Row count matches extraction":
        (
            exported_data_check.shape[0]
            == raw_manufacturing_data.shape[0]
        ),

    "Column count matches extraction":
        (
            exported_data_check.shape[1]
            == raw_manufacturing_data.shape[1]
        ),

    "File is below 5 MB":
        file_size_mb < 5
})

export_validation

File exists                         True
File size in MB                     2.19
Rows in exported CSV               10117
Columns in exported CSV               31
Row count matches extraction        True
Column count matches extraction     True
File is below 5 MB                  True
dtype: object

In [13]:
connection.close()

print("Database connection closed.")

Database connection closed.
